# Earthkit Climate Performance Analysis (SSP585)

This notebook reproduces the performance analysis of `earthkit-climate` indicators using the **SSP5-8.5** future projection dataset.

## Objectives
1.  Demonstrate the performance bottleneck in "Lazy" mode (Calculating percentiles on-the-fly).
2.  Demonstrate the optimization strategy (Persisting baselines + Chunking).
3.  Compare execution times for key indicators: **Warm Spell Duration Index (WSDI)**, **Consecutive Wet Days (CWD)**, and others.


In [1]:
import time
import cProfile
import pstats
import io

import earthkit.data
import xarray as xr
import dask.array as da

from earthkit.climate.indicators.precipitation import (
    daily_precipitation_intensity,
    maximum_consecutive_wet_days,
)
from earthkit.climate.indicators.temperature import (
    daily_temperature_range,
    heating_degree_days,
    warm_spell_duration_index,
)
from earthkit.climate.utils.percentile import percentile_doy

# Data URLs (Access-CM2)
URLS = {
    "pr_hist": "https://sites.ecmwf.int/repository/earthkit-climate/pr_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
    "tasmax_hist": "https://sites.ecmwf.int/repository/earthkit-climate/tasmax_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_historical.nc",
    "pr_ssp": "https://sites.ecmwf.int/repository/earthkit-climate/pr_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
    "tasmin_ssp": "https://sites.ecmwf.int/repository/earthkit-climate/tasmin_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
    "tasmax_ssp": "https://sites.ecmwf.int/repository/earthkit-climate/tasmax_gridded_day_CMIP6_ACCESS-CM2_r1i1p1f1_deepESD_day_ssp585.nc",
}

ModuleNotFoundError: No module named 'earthkit.climate'

## 1. Data Loading

We load the **SSP585** datasets for the analysis period and the **Historical** `tasmax` dataset to serve as the baseline for percentile calculations.

In [ ]:
def load_datasets():
    print("Loading Earthkit objects...")
    # Historical (Reference)
    tasmax_hist_ek = earthkit.data.from_source("url", URLS["tasmax_hist"])
    
    # SSP585 (Analysis)
    pr_ssp_ek = earthkit.data.from_source("url", URLS["pr_ssp"])
    tasmin_ssp_ek = earthkit.data.from_source("url", URLS["tasmin_ssp"])
    tasmax_ssp_ek = earthkit.data.from_source("url", URLS["tasmax_ssp"])
    
    print("Converting to Xarray...")
    tasmax_hist_ds = tasmax_hist_ek.to_xarray()
    pr_ds = pr_ssp_ek.to_xarray()
    tasmin_ds = tasmin_ssp_ek.to_xarray()
    tasmax_ds = tasmax_ssp_ek.to_xarray()
    
    # Metadata
    print(f"SSP585 Dimensions: {tasmax_ds.dims}")
    print(f"SSP585 Size (est): {tasmax_ds.nbytes / (1024*1024):.2f} MB")
    
    # Units fix
    tas_da = (tasmax_ds["tasmax"] + tasmin_ds["tasmin"]) / 2
    tas_da.attrs["units"] = tasmax_ds["tasmax"].attrs.get("units", "K")
    tas_ds = tas_da.to_dataset(name="tas")
    
    return tasmax_hist_ds, pr_ds, tasmin_ds, tasmax_ds, tas_ds

tasmax_hist_ds, pr_ds, tasmin_ds, tasmax_ds, tas_ds = load_datasets()

## 2. Profiling Helper

We define a helper to profile execution time. Crucially, it calls `.compute()` on the result to ensure lazy Dask graphs are evaluated.

In [ ]:
def profile_indicator(name, func, **kwargs):
    print(f"--- Profiling {name} ---")
    start = time.perf_counter()
    try:
        result = func(**kwargs)
        if hasattr(result, "to_xarray"):
            out = result.to_xarray()
            # Force computation
            if hasattr(out, "compute"):
                out.compute()
    except Exception as e:
        print(f"Error: {e}")
        return None
        
    elapsed = time.perf_counter() - start
    print(f"Time: {elapsed:.4f} seconds")
    return elapsed

## 3. Case Study: Unoptimized (Lazy) Execution

In this scenario:
1.  **Lazy Percentile**: `tasmax_per` is defined as a lazy dask graph.
2.  **Default Chunking**: We do not enforce contiguous time chunks.

This represents the naive usage pattern.

In [ ]:
print("Preparing Unoptimized Inputs...")

# 1. Define Lazy Percentile (No compute!)
tasmax_per_lazy = percentile_doy(tasmax_hist_ds["tasmax"], per=90)
tasmax_per_lazy.name = "tasmax_per"

# 2. Merge without re-chunking
wsdi_input_lazy = xr.merge([tasmax_ds, tasmax_per_lazy])
pr_input_lazy = pr_ds

print("Running Unoptimized Profiles...")
time_wsdi_lazy = profile_indicator("WSDI (Lazy)", warm_spell_duration_index, ds=wsdi_input_lazy)
time_cwd_lazy = profile_indicator("CWD (Lazy)", maximum_consecutive_wet_days, ds=pr_input_lazy)

## 4. Case Study: Optimized Execution

In this scenario:
1.  **Persisted Percentile**: We call `.compute()` on `tasmax_per` *before* passing it to the indicator. This separates the baseline calculation cost.
2.  **Optimized Chunking**: We re-chunk inputs with `{"time": -1}` to ensure run-length operations happen on contiguous blocks.

In [ ]:
print("Preparing Optimized Inputs...")

# 1. Pre-compute Percentile
print("Pre-computing 90th percentile from Historical data...")
start_per = time.perf_counter()
tasmax_per_opt = percentile_doy(tasmax_hist_ds["tasmax"], per=90)
tasmax_per_opt.name = "tasmax_per"
tasmax_per_opt = tasmax_per_opt.compute() # <--- CRITICAL OPTIMIZATION
print(f"Percentile computed in {time.perf_counter() - start_per:.2f}s")

# 2. Re-chunk for Time Contiguity
print("Re-chunking datasets...")
tasmax_ds_opt = tasmax_ds.chunk({"time": -1}) # <--- CRITICAL OPTIMIZATION for run-length
wsdi_input_opt = xr.merge([tasmax_ds_opt, tasmax_per_opt])

pr_input_opt = pr_ds.chunk({"time": -1})

print("Running Optimized Profiles...")
time_wsdi_opt = profile_indicator("WSDI (Optimized)", warm_spell_duration_index, ds=wsdi_input_opt)
time_cwd_opt = profile_indicator("CWD (Optimized)", maximum_consecutive_wet_days, ds=pr_input_opt)

## 5. Comparative Results

Summary of the speedup achieved on the **SSP585 (40-year)** dataset.

In [ ]:
import pandas as pd

results = {
    "Indicator": ["WSDI", "CWD"],
    "Lazy Time (s)": [time_wsdi_lazy, time_cwd_lazy],
    "Optimized Time (s)": [time_wsdi_opt, time_cwd_opt],
}

df = pd.DataFrame(results)
df["Speedup Factor"] = df["Lazy Time (s)"] / df["Optimized Time (s)"]
print(df)